In [ ]:
import os
import json
import yaml
import socket
import traceback
import smtplib
import logging
from datetime import datetime, timezone
from email.message import EmailMessage
from dataclasses import dataclass
from typing import Optional, Dict, Any, Set, List

import pandas as pd
import soccerdata as sd
import soccerdata._common as common
import undetected_chromedriver as uc

from pymongo import MongoClient
from pymongo.errors import BulkWriteError
from pydantic import BaseModel

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger("whoscored_raw_events")

# Patch 1: ChromeDriver version
CHROME_MAJOR = 147

def patched_init_webdriver(self):
    opts = uc.ChromeOptions()
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--start-maximized")
    return uc.Chrome(options=opts, version_main=CHROME_MAJOR)

common.BaseSeleniumReader._init_webdriver = patched_init_webdriver

# Patch 2: WhoScored sometimes returns JSON wrapped in minimal HTML
def tolerant_json_load(fp, *args, **kwargs):
    content = fp.read()

    if isinstance(content, bytes):
        content = content.decode("utf-8", errors="ignore")

    content = content.strip()

    if content.startswith("<html"):
        start = content.find("{")
        end = content.rfind("}") + 1
        if start >= 0 and end > start:
            content = content[start:end]

    return json.loads(content)

json.load = tolerant_json_load

logger.info("Patches loaded.")


In [ ]:
# 4. Config dataclasses
class MongoConfig(BaseModel):
    db: str
    url: str
    collection: Dict[str, str]

class SeasonConfig(BaseModel):
    year: str
    league: str
    name: str
    country: str

class EmailConfig(BaseModel):
    smtp_host: str
    smtp_port: int
    username: str
    from_email: str
    to_email: str
    use_tls: bool = True
    password: Optional[str] = None

class ScrapeDataConfig(BaseModel):
    mongo: MongoConfig
    season: SeasonConfig
    email: EmailConfig

def load_config(path: str, *, smtp_password_env: str = "SMTP_PASSWORD") -> ScrapeDataConfig:
    with open(path, "r") as f:
        raw = yaml.safe_load(f)

    email_cfg = EmailConfig(
        **raw["email"],
        password=os.getenv(smtp_password_env)
    )

    return ScrapeDataConfig(
        mongo=MongoConfig(**raw["mongo"]),
        season=SeasonConfig(**raw["season"]),
        email=email_cfg,
    )


events_config = load_config("../config/config.yaml")
events_config


In [ ]:
class ReportError:
    def __init__(self, config: ScrapeDataConfig):
        self.config = config

        self.client = MongoClient(config.mongo.url)
        self.db = self.client[config.mongo.db]

        logs_collection_name = config.mongo.collection["collection_logs"]
        self.collection = self.db[logs_collection_name]

    def report(self, job_name: str, exc: Exception, context: Optional[Dict[str, Any]] = None) -> None:
        context = context or {}

        error_doc = {
            "job": job_name,
            "timestamp": datetime.now(timezone.utc),
            "host": socket.gethostname(),
            "season_year": self.config.season.year,
            "league": self.config.season.league,
            "competition": self.config.season.name,
            "country": self.config.season.country,
            "context": context,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
            "traceback": traceback.format_exc(),
        }

        try:
            self.collection.insert_one(error_doc)
            logger.info("Error saved to MongoDB")
        except Exception:
            logger.exception("Failed to save error to MongoDB")

        if self.config.email.password:
            self._send_email(error_doc)
        else:
            logger.warning("Email not sent: SMTP_PASSWORD is missing")

    def _send_email(self, error_doc: dict) -> None:
        cfg = self.config.email

        subject = f"[WhoScored Scrape ERROR] {error_doc['job']} | {self.config.season.name} {self.config.season.year}"
        body = (
            f"Job: {error_doc['job']}\n"
            f"Time (UTC): {error_doc['timestamp']}\n"
            f"Host: {error_doc['host']}\n\n"
            f"League/Season: {self.config.season.league} / {self.config.season.year}\n"
            f"Competition: {self.config.season.name} ({self.config.season.country})\n\n"
            f"Context: {error_doc['context']}\n\n"
            f"{error_doc['error_type']}: {error_doc['error_message']}\n\n"
            f"Traceback:\n{error_doc['traceback']}\n"
        )

        msg = EmailMessage()
        msg["Subject"] = subject
        msg["From"] = cfg.from_email
        msg["To"] = cfg.to_email
        msg.set_content(body)

        try:
            if cfg.use_tls:
                with smtplib.SMTP(cfg.smtp_host, cfg.smtp_port, timeout=20) as server:
                    server.starttls()
                    server.login(cfg.username, cfg.password)
                    server.send_message(msg)
            else:
                with smtplib.SMTP_SSL(cfg.smtp_host, cfg.smtp_port, timeout=20) as server:
                    server.login(cfg.username, cfg.password)
                    server.send_message(msg)

            logger.info("Error email sent")

        except Exception:
            logger.exception("Failed to send error email")


reporter = ReportError(config=events_config)


In [ ]:
client = MongoClient(events_config.mongo.url)
db = client[events_config.mongo.db]

logger.info("Mongo DB: %s", db.name)
logger.info("Collections: %s", db.list_collection_names())


In [ ]:
class RawEvents:
    def __init__(self, config: ScrapeDataConfig, reporter: ReportError):
        self.config = config
        self.reporter = reporter

        self.client = MongoClient(config.mongo.url)
        self.db = self.client[config.mongo.db]

        schedule_name = config.mongo.collection["collection_schedule"]
        self.collection_schedule = self.db[schedule_name]

        raw_events_name = config.mongo.collection["collection_raw_events"]
        self.collection_raw_events = self.db[raw_events_name]

    def _finished_games_df(self) -> pd.DataFrame:
        docs = list(
            self.collection_schedule.find(
                {
                    "season": self.config.season.year,
                    "game_status": "finished",
                },
                {"_id": 0},
            )
        )
        return pd.DataFrame(docs)

    def _raw_events_extracted(self) -> Set[int]:
        return {int(x) for x in self.collection_raw_events.distinct("game_id") if x is not None}

    @staticmethod
    def _normalize_mongo_value(value):
        if value is None:
            return None

        try:
            if pd.isna(value):
                return None
        except Exception:
            pass

        if isinstance(value, pd.Timestamp):
            return value.to_pydatetime()

        return value

    def _process_raw_events(self, game_info: pd.DataFrame, events: pd.DataFrame) -> List[dict]:
        events = events.reset_index(drop=False)

        if "event_idx" not in events.columns:
            if "index" in events.columns:
                events = events.rename(columns={"index": "event_idx"})
            else:
                events["event_idx"] = range(len(events))

        for col in ["league", "season", "game"]:
            if col in events.columns:
                events = events.drop(columns=[col])

        info = game_info.copy()

        info["game_id"] = info["game_id"].astype(int)
        events["game_id"] = events["game_id"].astype(int)

        merged = pd.merge(info, events, on="game_id", how="right")

        records = []
        for record in merged.to_dict(orient="records"):
            clean = {k: self._normalize_mongo_value(v) for k, v in record.items()}
            records.append(clean)

        return records

    def save_new_finished_games(self, limit: Optional[int] = None) -> None:
        job_name = "RawEvents.save_new_finished_games"

        try:
            games_df = self._finished_games_df()

            if games_df.empty:
                logger.info("No finished games found in schedule collection.")
                return

            extracted_game_ids = self._raw_events_extracted()

            game_ids = [int(x) for x in games_df["game_id"].tolist()]
            to_process = [gid for gid in game_ids if gid not in extracted_game_ids]

            if limit is not None:
                to_process = to_process[:limit]

            if not to_process:
                logger.info("No new games to process. Raw events already present.")
                return

            logger.info(
                "Games to process: %d | Already saved: %d",
                len(to_process),
                len(game_ids) - len(to_process),
            )

            games_info = games_df[games_df["game_id"].isin(to_process)].reset_index(drop=True)

            ws = sd.WhoScored(
                leagues=self.config.season.league,
                seasons=self.config.season.year,
            )

            inserted_games = 0
            inserted_rows_total = 0

            for game_id in to_process:
                game_id = int(game_id)

                try:
                    logger.info("Reading events for game_id=%s", game_id)

                    events = ws.read_events(match_id=game_id, output_fmt="events")

                    if events is None or len(events) == 0:
                        logger.info("No events for game_id=%s. Skipping.", game_id)
                        continue

                    game_info = games_info[games_info["game_id"] == game_id].reset_index(drop=True)

                    if game_info.empty:
                        logger.warning("No schedule metadata found for game_id=%s. Skipping.", game_id)
                        continue

                    records = self._process_raw_events(game_info=game_info, events=events)

                    if not records:
                        logger.info("No merged records for game_id=%s. Skipping.", game_id)
                        continue

                    try:
                        result = self.collection_raw_events.insert_many(records, ordered=False)
                        inserted_count = len(result.inserted_ids)
                    except BulkWriteError as bwe:
                        inserted_count = bwe.details.get("nInserted", 0)
                        logger.warning(
                            "BulkWriteError for game_id=%s. Inserted rows before duplicates/errors: %s",
                            game_id,
                            inserted_count,
                        )

                    inserted_games += 1
                    inserted_rows_total += inserted_count

                    logger.info("Inserted game_id=%s | rows=%d", game_id, inserted_count)

                except Exception as e:
                    self.reporter.report(
                        job_name="RawEvents.insert_game",
                        exc=e,
                        context={
                            "game_id": game_id,
                            "league": self.config.season.league,
                            "season": self.config.season.year,
                        },
                    )
                    logger.exception("Error processing game_id=%s", game_id)

            logger.info(
                "Raw events complete. Games inserted: %d | Total rows inserted: %d",
                inserted_games,
                inserted_rows_total,
            )
            client.close()
        except Exception as e:
            self.reporter.report(
                job_name=job_name,
                exc=e,
                context={
                    "season_year": self.config.season.year,
                    "league": self.config.season.league,
                },
            )
            raise


In [ ]:
raw_events = RawEvents(config=events_config, reporter=reporter)

games_df = raw_events._finished_games_df()
already_done = raw_events._raw_events_extracted()

raw_events = RawEvents(config=events_config, reporter=reporter)

games_df = raw_events._finished_games_df()
already_done = raw_events._raw_events_extracted()

logger.info("Finished games in schedule: %d", len(games_df))
logger.info("Games already in raw_events: %d", len(already_done))

if not games_df.empty:
    pending = games_df[~games_df["game_id"].astype(int).isin(already_done)].copy()
    logger.info("Pending games: %d", len(pending))
    display(pending.head())
logger.info("Games already in raw_events: %d", len(already_done))

if not games_df.empty:
    pending = games_df[~games_df["game_id"].astype(int).isin(already_done)].copy()
    logger.info("Pending games: %d", len(pending))
    display(pending.head())


In [ ]:
raw_events.save_new_finished_games()

In [ ]:
client = MongoClient(events_config.mongo.url)
df = pd.DataFrame(client['WhoScored'].game_raw_events.find({'season': events_config.season.year}))
# df = pd.DataFrame(client['WhoScored'].game_raw_events.find({}))

In [ ]:
df.groupby('season')["game_id"].nunique()

In [ ]:
df["game_id"].value_counts()